# 第三阶段：模型改进与消融实验
方案A：SAHI切片推理 | 方案B：CBAM注意力 | 方案C：小目标检测头

## 方案A：SAHI 切片推理

In [ ]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

model = AutoDetectionModel.from_pretrained(
    'ultralytics',
    model_path='results/baseline_v1/weights/best.pt',
    confidence_threshold=0.3,
)
result = get_sliced_prediction(
    'data/VisDrone/VisDrone2019-DET-val/images/0000001_00000_d_0000001.jpg',
    model,
    slice_height=256, slice_width=256,
    overlap_height_ratio=0.2, overlap_width_ratio=0.2,
)
result.export_visuals(export_dir='results/sahi/')
print(f'检测到 {len(result.object_prediction_list)} 个目标')

## 方案B：YOLOv11 + CBAM 训练

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
model.train(
    data='configs/visdrone.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    name='cbam',
    project='results',
    plots=True,
)

## 消融实验汇总

In [ ]:
import pandas as pd
from ultralytics import YOLO

experiments = {
    'Baseline':        'results/baseline_v1/weights/best.pt',
    'Baseline+SAHI':   'results/baseline_v1/weights/best.pt',
    'CBAM':            'results/cbam/weights/best.pt',
}

rows = []
for name, weights in experiments.items():
    m = YOLO(weights)
    if 'SAHI' in name:
        rows.append({'Model': name, 'mAP50': '-', 'mAP50-95': '-', '备注': '见SAHI单元格'})
        continue
    met = m.val(data='configs/visdrone.yaml', imgsz=640, verbose=False)
    rows.append({'Model': name, 'mAP50': f'{met.box.map50:.4f}', 'mAP50-95': f'{met.box.map:.4f}', '备注': ''})

df = pd.DataFrame(rows)
print(df.to_string(index=False))